In [1]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')

In [3]:
# Load env variables and create client

from anthropic import Anthropic
from pathlib import Path

client = Anthropic(
    api_key=my_anthropic_key,
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = my_anthropic_model


In [4]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [5]:
file_metadata = upload("streaming.csv")
file_metadata

BetaFileMetadata(id='file_01A1Zm35CakHfGC9xEURFg76', created_at=datetime.datetime(2026, 9, 17, 7, 5, 50, 131491, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, expires_at=None, scope=None)

In [6]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate.
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

Message(id='msg_011Cf8bB9WMb8MnRzkEFBHy6', container=Container(id='container_01MbWCg3tzRaH1r1E5kpT7DV', expires_at=datetime.datetime(2026, 9, 17, 8, 11, 47, 96407, tzinfo=TzInfo(0)), skills=None), content=[TextBlock(citations=None, text="I'll analyze the uploaded streaming dataset to determine the major drivers of churn. Let me start by exploring the data.", type='text'), ServerToolUseBlock(id='srvtoolu_014o41o83AQXjHR3zpbkJn3s', caller=None, input={'command': 'view', 'path': '$INPUT_DIR'}, name='text_editor_code_execution', type='server_tool_use'), TextEditorCodeExecutionToolResultBlock(content=TextEditorCodeExecutionToolResultError(error_code='unavailable', error_message='Tool response parsing error for view: Failed to parse tool response as JSON: unexpected character: line 1 column 1 (char 0)', type='text_editor_code_execution_tool_result_error'), tool_use_id='srvtoolu_014o41o83AQXjHR3zpbkJn3s', type='text_editor_code_execution_tool_result'), TextBlock(citations=None, text='Let me c

In [8]:
download_file("file_01H4YZWe91cjS39GbYYjimx8")